# Raw → Silver

This notebook cleans the three bronze tables (`crm_accounts`, `organizations`, `signins`) and writes them to silver.

**Steps, one per section:**
1. **Declare variables** — all table names and lookup values, in plain sight at the top
2. **Read** the three bronze tables
3. **Convert data types** — text → timestamps / ints, trim spaces, standardize country codes
4. **Run data quality checks and key validations** — nulls, duplicate keys, allowed values, foreign keys. Each check prints PASS or FAIL.
5. **Write** the cleaned tables to silver

## 1. Variables

Everything the notebook needs is declared here. To point this notebook at different tables, change only this cell.

In [0]:
from pyspark.sql import functions as F

# --- table names (bronze = raw source, silver = cleaned output) ---
catalog = "coreview"

accounts_bronze = f"{catalog}.bronze.crm_accounts"
accounts_silver = f"{catalog}.silver.crm_accounts"

organizations_bronze = f"{catalog}.bronze.organizations"
organizations_silver = f"{catalog}.silver.organizations"

signins_bronze = f"{catalog}.bronze.signins"
signins_silver = f"{catalog}.silver.signins"

# --- lookup for the messy free-text 'country' column ---
# key = the value after trim + lowercase, value = the standard country code we want
country_codes = {
    "uk": "UK", "gb": "UK", "united kingdom": "UK",
    "us": "US", "usa": "US", "united states": "US", "united states of america": "US",
    "canada": "CA", "ca": "CA",
    "france": "FR", "fr": "FR",
    "australia": "AU", "au": "AU",
    "germany": "DE", "de": "DE",
}

# --- timestamp formats that appear in the data ---
ts_format = "yyyy-MM-dd HH:mm:ss"        # e.g. 2026-11-18 09:21:34
iso_format = "yyyy-MM-dd'T'HH:mm:ss'Z'"  # e.g. 2026-01-01T02:44:49Z

# --- allowed values for the check section ---
#allowed_segments = ["SMB", "Mid-Market", "Enterprise"]
allowed_statuses = ["active", "cancelled"]
#allowed_regions = ["US", "EU", "CA", "AU"]
allowed_results = ["success", "failure"]

## 2. Read the bronze tables

if the pipeline was incremental we would have just read the latest records

In [0]:
# def read_incremental(bronze_df, silver_table, timestamp_col="Ingestion_timestamp"):
#     """Read only the latest records: if the silver table already exists,
#     keep only bronze rows whose ingestion timestamp is newer than the max
#     already in silver. If silver does not exist yet, read the full table."""
#     if spark.catalog.tableExists(silver_table):
#         max_ts = spark.table(silver_table).agg(F.max(timestamp_col)).collect()[0][0]
#         if max_ts is not None:
#             print(f"incremental load from bronze -> {silver_table}: only rows after {max_ts}")
#             return bronze_df.where(F.col(timestamp_col).cast("timestamp") > F.lit(max_ts))
#     print(f"full load (silver table {silver_table} not found or empty)")
#     return bronze_df


# df_accounts = read_incremental(spark.table(accounts_bronze), accounts_silver)
# df_organizations = read_incremental(spark.table(organizations_bronze), organizations_silver)
# df_signins = read_incremental(spark.table(signins_bronze), signins_silver)

# print(f"crm_accounts:    {df_accounts.count()} rows")
# print(f"organizations:   {df_organizations.count()} rows")
# print(f"signins:        {df_signins.count()} rows")

In [0]:
df_accounts = spark.table(accounts_bronze)
df_organizations = spark.table(organizations_bronze)
df_signins = spark.table(signins_bronze)

print(f"crm_accounts:    {df_accounts.count()} rows")
print(f"organizations:   {df_organizations.count()} rows")
print(f"signins:        {df_signins.count()} rows")

crm_accounts:    180 rows
organizations:   12125 rows
signins:        302593 rows


## 3. Data type conversions and cleaning

Five small helpers, each doing **one** simple thing. They work on any table — you pass the DataFrame and the column names as parameters.

In [0]:
def trim_spaces(df, columns):
    """Remove leading/trailing spaces from the given text columns."""
    for c in columns:
        df = df.withColumn(c, F.trim(c))
    return df


def convert_to_timestamp(df, column, formats):
    """Convert a text column to a real timestamp, trying each format in order.
    A value that matches no format becomes null — the checks below will catch it."""
    parsed = F.coalesce(*[F.try_to_timestamp(F.col(column), F.lit(f)) for f in formats])
    return df.withColumn(column, parsed)


def convert_to_int(df, column):
    """Convert a text column to a real int. Bad values become null."""
    return df.withColumn(column, F.col(column).cast("int"))


def standardize_values(df, column, lookup):
    """Replace messy text values with standard ones from a lookup dict.
    Matches on the trimmed, lower-case value; anything not in the lookup stays as-is (trimmed)."""
    pairs = [F.lit(x) for key, value in lookup.items() for x in (key, value)]
    value_map = F.create_map(*pairs)
    return df.withColumn(
        column,
        F.coalesce(F.element_at(value_map, F.lower(F.trim(column))), F.trim(column)),
    )

### Derived Columns

In [0]:
def derive_crm_key(df, account_col, length, new_col):
    """Add a 'crm_key' column: the first 15 characters of the account_id."""
    return df.withColumn(new_col, F.lower(F.substring(F.col(account_col), 1, length)))


def derive_capture_date(df, source_col, new_col):
    """Extract the date from a source file name like '2026-02-24.jsonl'
    and add it as a real date column (2026-02-24)."""
    return df.withColumn(new_col, F.to_date(F.regexp_extract(F.col(source_col), r"(\d{4}-\d{2}-\d{2})", 1)))


def derive_is_test(df, tenant_col, new_col):
    """Flag test/demo tenants: True if the (lower-cased) tenant name starts with 'test' or 'demo'."""
    lowered = F.lower(F.col(tenant_col))
    return df.withColumn(new_col, lowered.startswith("test") | lowered.startswith("demo"))


In [0]:
# --- crm_accounts ---
df_accounts = trim_spaces(df_accounts, ["account_id", "account_name", "segment"])
df_accounts = standardize_values(df_accounts, "country", country_codes)  # ' usa ', 'GB', 'UK', ... -> 'US', 'UK', ...
df_accounts = derive_crm_key(df_accounts, "account_id", 15, "crm_key")

display(df_accounts.limit(10))

account_id,account_name,country,segment,Ingestion_timestamp,Ingestion_date,crm_key
001FXqc67PSEoGvPIC,Harbor Insurance,UK,SMB,2026-11-18T21:56:22.157Z,2026-11-18,001fxqc67pseogv
001PV6mIS5NmL0yUXZ,Juniper Finance,FR,Enterprise,2026-11-18T21:56:22.157Z,2026-11-18,001pv6mis5nml0y
001IjM8z1QOPEY9YHQ,Lakeside Media,AU,Mid-Market,2026-11-18T21:56:22.157Z,2026-11-18,001ijm8z1qopey9
0019SAmry7HvhlUSWU,Falcon Partners,UK,SMB,2026-11-18T21:56:22.157Z,2026-11-18,0019samry7hvhlu
0015KugxFSxBaRFLDJ,Westgate Finance,FR,Mid-Market,2026-11-18T21:56:22.157Z,2026-11-18,0015kugxfsxbarf
001gbKHEKRgzoBVIXU,Redstone Labs,DE,Mid-Market,2026-11-18T21:56:22.157Z,2026-11-18,001gbkhekrgzobv
001cThBZRvuCrQzJEJ,Starling Clinics,US,Enterprise,2026-11-18T21:56:22.157Z,2026-11-18,001cthbzrvucrqz
001q9d3zuPTORaJKCX,Granite Energy,DE,Mid-Market,2026-11-18T21:56:22.157Z,2026-11-18,001q9d3zuptoraj
001ZqKnRKNABsiJLJJ,Oakridge Builders,CA,Mid-Market,2026-11-18T21:56:22.157Z,2026-11-18,001zqknrknabsij
001fqkdjoiLovRyMHL,Juniper Health,DE,SMB,2026-11-18T21:56:22.157Z,2026-11-18,001fqkdjoilovry


In [0]:
# --- duplicate check on df_accounts ---
total = df_accounts.count()
distinct_ids = df_accounts.select("account_id").distinct().count()
dupe_rows = total - distinct_ids
print(f"total rows:              {total}")
print(f"distinct account_ids:     {distinct_ids}")
print(f"duplicate account_ids:    {dupe_rows}")

key_dupes = (df_accounts
             .groupBy("account_id", "account_name", "country", "segment")
             .count()
             .filter(F.col("count") > 1))

print(f"account_ids appearing more than once: {key_dupes.count()}")

display(key_dupes.orderBy(F.col("count").desc()).limit(20))


total rows:              180
distinct account_ids:     180
duplicate account_ids:    0
account_ids appearing more than once: 0


account_id,account_name,country,segment,count


In [0]:
df_accounts = df_accounts.dropDuplicates(["account_id", "account_name", "country", "segment"])

df_accounts.count()

180

In [0]:

df_organizations = trim_spaces(df_organizations, ["crm_ref", "tenant_name", "license_plan", "region", "status"])
df_organizations = df_organizations.withColumn("crm_ref", F.lower("crm_ref"))
df_organizations = convert_to_timestamp(df_organizations, "created_at", [iso_format])
df_organizations = convert_to_int(df_organizations, "license_seats")

df_organizations = derive_capture_date(df_organizations, "source_file_name", "capture_date")
df_organizations = derive_is_test(df_organizations, "tenant_name", "is_test")


display(df_organizations.limit(10))

created_at,crm_ref,license_addons,license_plan,license_seats,org_id,region,status,tenant_name,source_file_name,Ingestion_timestamp,Ingestion_date,capture_date,is_test
2025-03-25T17:34:00.000Z,001fxqc67pseogvpic,List(audit),starter,1000,0070366f29d4baa30aecd4bf,US,active,Harbor Insurance AU,2026-02-24.jsonl,2026-11-18T21:56:30.845Z,2026-11-18,2026-02-24,false
2025-03-17T00:31:00.000Z,001juw1swaw7uxz,List(backup),standard,100,0367b50f9264c4019b39fc28,CA,active,Highland Legal,2026-02-24.jsonl,2026-11-18T21:56:30.845Z,2026-11-18,2026-02-24,false
2025-10-11T16:53:00.000Z,001uyft0stzozpa,"List(audit, reports)",starter,10,040f2aec3fcc9a8ea7d33a51,AU,active,Westgate Retail,2026-02-24.jsonl,2026-11-18T21:56:30.845Z,2026-11-18,2026-02-24,false
2024-10-01T01:23:00.000Z,001mihb18tuzeok,"List(audit, backup)",enterprise,500,04a836d16bf160863057a5f6,AU,active,Starling Energy,2026-02-24.jsonl,2026-11-18T21:56:30.845Z,2026-11-18,2026-02-24,false
2024-08-02T17:49:00.000Z,0013frr0tinrkob,List(),starter,500,07455d260415f3ed1a9f7130,AU,active,Juniper Logistics,2026-02-24.jsonl,2026-11-18T21:56:30.845Z,2026-11-18,2026-02-24,false
2026-01-27T00:57:00.000Z,0019rawhwircua0,"List(backup, reports)",starter,2000,075e8f602fb1775b362e67fb,US,active,Summit Health,2026-02-24.jsonl,2026-11-18T21:56:30.845Z,2026-11-18,2026-02-24,false
2024-08-17T13:12:00.000Z,001rz3hh3w7i8y1,"List(audit, reports)",standard,50,091c578e9022ba711b0b9f25,CA,active,Sterling Foods,2026-02-24.jsonl,2026-11-18T21:56:30.845Z,2026-11-18,2026-02-24,false
2025-03-02T12:13:00.000Z,0011zzhpewgpakh,"List(audit, reports)",enterprise,50,094cc970cd7cf30aa30ff992,CA,active,Cedar Foods,2026-02-24.jsonl,2026-11-18T21:56:30.845Z,2026-11-18,2026-02-24,false
2024-02-02T02:16:00.000Z,001g3rn2nfz5b4ubsa,"List(audit, backup)",standard,100,0a495cb327b4d82cf8e9ebc0,EU,active,Cedar Media,2026-02-24.jsonl,2026-11-18T21:56:30.845Z,2026-11-18,2026-02-24,false
2026-02-08T09:49:00.000Z,0015kugxfsxbarfldj,List(audit),starter,500,0c0faa81bc5d2e473d8981a7,AU,active,Westgate Finance,2026-02-24.jsonl,2026-11-18T21:56:30.845Z,2026-11-18,2026-02-24,false


In [0]:
# --- duplicate check on df_organizations ---
total = df_organizations.count()
distinct_full_rows = df_organizations.distinct().count()
full_row_dupes = total - distinct_full_rows
print(f"total rows:                {total}")
print(f"distinct rows (all cols):  {distinct_full_rows}")
print(f"exact duplicate rows:      {full_row_dupes}")

# duplicates by key (org_id is expected to repeat: one row per license addon)
key_dupes = (df_organizations
             .groupBy("created_at", "crm_ref", "license_plan","license_seats", "org_id", "tenant_name", "region", "status", "capture_date")
             .count()
             .filter(F.col("count") > 1))
print(f"org_ids appearing more than once per day: {key_dupes.count()}")

display(key_dupes.orderBy(F.col("count").desc()).limit(20))

total rows:                12125
distinct rows (all cols):  12125
exact duplicate rows:      0
org_ids appearing more than once per day: 0


created_at,crm_ref,license_plan,license_seats,org_id,tenant_name,region,status,capture_date,count


In [0]:
df_organizations = df_organizations.drop("license_addons")

df_organizations = df_organizations.dropDuplicates(["created_at", "crm_ref", "license_plan","license_seats", "org_id", "tenant_name", "region", "status", "capture_date"])

df_organizations.count()

12125

In [0]:
display(df_organizations)

created_at,crm_ref,license_plan,license_seats,org_id,region,status,tenant_name,source_file_name,Ingestion_timestamp,Ingestion_date,capture_date,is_test
2025-03-25T17:34:00.000Z,001fxqc67pseogvpic,starter,1000,0070366f29d4baa30aecd4bf,US,active,Harbor Insurance AU,2026-02-24.jsonl,2026-11-18T21:56:30.845Z,2026-11-18,2026-02-24,false
2025-03-17T00:31:00.000Z,001juw1swaw7uxz,standard,100,0367b50f9264c4019b39fc28,CA,active,Highland Legal,2026-02-24.jsonl,2026-11-18T21:56:30.845Z,2026-11-18,2026-02-24,false
2025-10-11T16:53:00.000Z,001uyft0stzozpa,starter,10,040f2aec3fcc9a8ea7d33a51,AU,active,Westgate Retail,2026-02-24.jsonl,2026-11-18T21:56:30.845Z,2026-11-18,2026-02-24,false
2024-10-01T01:23:00.000Z,001mihb18tuzeok,enterprise,500,04a836d16bf160863057a5f6,AU,active,Starling Energy,2026-02-24.jsonl,2026-11-18T21:56:30.845Z,2026-11-18,2026-02-24,false
2024-08-02T17:49:00.000Z,0013frr0tinrkob,starter,500,07455d260415f3ed1a9f7130,AU,active,Juniper Logistics,2026-02-24.jsonl,2026-11-18T21:56:30.845Z,2026-11-18,2026-02-24,false
2026-01-27T00:57:00.000Z,0019rawhwircua0,starter,2000,075e8f602fb1775b362e67fb,US,active,Summit Health,2026-02-24.jsonl,2026-11-18T21:56:30.845Z,2026-11-18,2026-02-24,false
2024-08-17T13:12:00.000Z,001rz3hh3w7i8y1,standard,50,091c578e9022ba711b0b9f25,CA,active,Sterling Foods,2026-02-24.jsonl,2026-11-18T21:56:30.845Z,2026-11-18,2026-02-24,false
2025-03-02T12:13:00.000Z,0011zzhpewgpakh,enterprise,50,094cc970cd7cf30aa30ff992,CA,active,Cedar Foods,2026-02-24.jsonl,2026-11-18T21:56:30.845Z,2026-11-18,2026-02-24,false
2024-02-02T02:16:00.000Z,001g3rn2nfz5b4ubsa,standard,100,0a495cb327b4d82cf8e9ebc0,EU,active,Cedar Media,2026-02-24.jsonl,2026-11-18T21:56:30.845Z,2026-11-18,2026-02-24,false
2026-02-08T09:49:00.000Z,0015kugxfsxbarfldj,starter,500,0c0faa81bc5d2e473d8981a7,AU,active,Westgate Finance,2026-02-24.jsonl,2026-11-18T21:56:30.845Z,2026-11-18,2026-02-24,false


In [0]:
# --- signins ---
# event_ts arrives in two different formats — the helper simply tries both
df_signins = convert_to_timestamp(df_signins, "event_ts", [iso_format, ts_format])

# remove rows that were loaded more than once
# df_signins = df_signins.dropDuplicates()

display(df_signins.limit(10))

event_id,org_id,user_id,event_ts,result,ip_hash,Ingestion_timestamp,Ingestion_date
53b004a4918a0831676362729880819e,1f380cb17357fa5aa402b2dd,812f284a0e0d2f5a,2026-01-01T02:44:49.000Z,success,0de01d62e0efbe27,2026-11-18 21:56:36,2026-11-18
553f18d8f0d1b161a47ef08a674b7c07,849e828455bb08497d00d31a,c7517976ae620759,2026-01-01T14:19:40.000Z,success,5987a77f8d6275f8,2026-11-18 21:56:36,2026-11-18
53294f327bb893d34734c14cf9085aeb,841d26cae27c56bdbe30855a,d3cc9fae0f20c54f,2026-01-01T08:18:26.000Z,success,ab998cdb9efd3ee2,2026-11-18 21:56:36,2026-11-18
6ff35b3a5dcc5bb561557226104eb5c5,e067bb267147d8037c5533d3,43f61b43e1269673,2026-01-01T02:38:55.000Z,success,cc561bf0ccf171c0,2026-11-18 21:56:36,2026-11-18
00779d3f92629ffbaad0f1fbab6b91f3,f6fbf93c0801a81329e35a08,a2b11407a02fd30a,2026-01-01T10:27:16.000Z,success,c63ba3c64c129ce0,2026-11-18 21:56:36,2026-11-18
d983be977441b944649f058ebc7bb454,d4ea5988bde5874f9b95b077,a987df92af5d612d,2026-01-01T14:19:03.000Z,success,8879f1187d67ea03,2026-11-18 21:56:36,2026-11-18
16991e4e8181cc8e891909ff907f4942,7a7b93c04ba0bd34f19d07d2,5eb471e7dc0873d7,2026-01-01T05:30:40.000Z,failure,fc4aa6b129cbd173,2026-11-18 21:56:36,2026-11-18
94f29f7f04d03d6c0d3075c60b36fcb2,f0ae89212b9c62397c2163e9,a02efc4456908cdd,2026-01-01T06:31:33.000Z,success,586653a208a341a7,2026-11-18 21:56:36,2026-11-18
027b9a0ba8d291f47698e0bd42c2de35,58cfe411499a3d1dd16f9015,bd7045ef01c597de,2026-01-01T22:16:30.000Z,success,38041e47b5a93647,2026-11-18 21:56:36,2026-11-18
ea883bf6ab44121c2d28c1da5dfebe0f,091c578e9022ba711b0b9f25,0023f19c149f764f,2026-01-01T08:36:07.000Z,success,e4ea4695f6bf2d63,2026-11-18 21:56:36,2026-11-18


In [0]:
# --- duplicate check on df_signins ---
total = df_signins.count()
distinct_full_rows = df_signins.distinct().count()
full_row_dupes = total - distinct_full_rows
print(f"total rows:                {total}")
print(f"distinct rows (all cols):  {distinct_full_rows}")
print(f"exact duplicate rows:      {full_row_dupes}")

# duplicates by key (event_id should be unique — each signin event appears once)
key_dupes = (df_signins
             .groupBy("event_id")
             .count()
             .filter(F.col("count") > 1))
print(f"event_ids appearing more than once: {key_dupes.count()}")

display(key_dupes.orderBy(F.col("count").desc()).limit(20))

total rows:                302593
distinct rows (all cols):  293842
exact duplicate rows:      8751
event_ids appearing more than once: 8751


event_id,count
dfdc4773eb9675a4416c285044a4dd22,2
f604386d3ab310316e9e5fafc20bd01b,2
9b3449d4166f2aa67fd0a03f91d6aa60,2
189e84be1bae700abe88e95f19bc3f95,2
2bc21f88f98435dbbd42ae32b758c0a3,2
ad5f37317dbf531a7bbc04de4119a9f1,2
1c21fde624b2bc76b69dd13927a69dac,2
0383a6b6ab365a88e30319d124f54b45,2
3c7a89e5fce7550ce906ab84ef073fb7,2
13186331882aedcf1661c906fc484191,2


In [0]:
df_signins = df_signins.dropDuplicates(["event_id", "event_ts", "org_id", "user_id", "ip_hash"])

## 4. Data quality checks & key validations

Four small helpers, each doing **one** check and printing PASS or FAIL with the number of bad rows. They work on any table — you pass the DataFrame, the table name (only for the printed message) and the columns as parameters.

In [0]:
def check_not_null(df, table_name, columns):
    """Every value in these columns must be filled in."""
    for c in columns:
        nulls = df.where(F.col(c).isNull()).count()
        result = "PASS" if nulls == 0 else f"FAIL - {nulls} null values"
        print(f"[{table_name}] {c} is not null: {result}")


def check_unique(df, table_name, key_columns):
    """These columns form the key — no two rows should share the same key."""
    total_rows = df.count()
    unique_rows = df.select(*key_columns).distinct().count()
    duplicates = total_rows - unique_rows
    result = "PASS" if duplicates == 0 else f"FAIL - {duplicates} duplicate rows"
    print(f"[{table_name}] {', '.join(key_columns)} is unique: {result}")


def check_allowed_values(df, table_name, column, allowed):
    """The column may only contain the values in the given list."""
    bad = df.where(F.col(column).isNull() | ~F.col(column).isin(allowed)).count()
    result = "PASS" if bad == 0 else f"FAIL - {bad} unexpected values"
    print(f"[{table_name}] {column} only has allowed values: {result}")


def check_references(df, table_name, column, ref_df, ref_column):
    """Foreign key check: every value in 'column' must exist in 'ref_column' of the reference table."""
    refs = ref_df.select(F.col(ref_column).alias("ref_value")).distinct()
    orphans = (df.where(F.col(column).isNotNull())
                 .join(refs, F.col(column) == F.col("ref_value"), "left_anti")
                 .count())
    result = "PASS" if orphans == 0 else f"FAIL - {orphans} rows point to a missing {ref_column}"
    print(f"[{table_name}] {column} exists in {ref_column}: {result}")

In [0]:
# --- crm_accounts ---
check_not_null(df_accounts, "crm_accounts", ["account_id"])
check_unique(df_accounts, "crm_accounts", ["account_id"])
# check_allowed_values(df_accounts, "crm_accounts", "segment", allowed_segments)
# check_allowed_values(df_accounts, "crm_accounts", "country", sorted(set(country_codes.values())))

[crm_accounts] account_id is not null: PASS
[crm_accounts] account_id is unique: PASS


In [0]:
# --- organizations ---
check_not_null(df_organizations, "organizations", ["org_id"])

# org_id alone is NOT unique on purpose: one organization gets one row per license
# addon (and we keep every version), so this FAIL is expected and fine.
check_unique(df_organizations, "organizations", ["org_id", "capture_date"])

# some organizations have no CRM link (null crm_ref) and a few point to accounts
# we don't have — both FAILs are known issues in the source data.
check_not_null(df_organizations, "organizations", ["crm_ref"])
check_references(df_organizations, "organizations", "crm_ref", df_accounts, "crm_key")

check_allowed_values(df_organizations, "organizations", "status", allowed_statuses)
# check_allowed_values(df_organizations, "organizations", "region", allowed_regions)

[organizations] org_id is not null: PASS
[organizations] org_id, capture_date is unique: PASS
[organizations] crm_ref is not null: FAIL - 826 null values
[organizations] crm_ref exists in crm_key: FAIL - 5890 rows point to a missing crm_key
[organizations] status only has allowed values: PASS


In [0]:
# --- signins ---
check_not_null(df_signins, "signins", ["event_id", "event_ts", "org_id"])
check_unique(df_signins, "signins", ["event_id", "org_id"])
check_allowed_values(df_signins, "signins", "result", allowed_results)

# known issue in the source data: some signins belong to organizations we have no record of
check_references(df_signins, "signins", "org_id", df_organizations, "org_id")

[signins] event_id is not null: PASS
[signins] event_ts is not null: PASS
[signins] org_id is not null: PASS
[signins] event_id, org_id is unique: PASS
[signins] result only has allowed values: PASS
[signins] org_id exists in org_id: FAIL - 500 rows point to a missing org_id


## 5. Write to silver

`overwrite` means running this cell again simply replaces the silver tables with the newest version.

would be using this if the pipeline is incremental

In [0]:
# def write_to_silver(df, silver_table):
#     """Append to the silver table if it already exists, otherwise create it fresh (overwrite)."""
#     mode = "append" if spark.catalog.tableExists(silver_table) else "overwrite"
#     print(f"{silver_table}: mode = {mode}")
#     df = (df.withColumn("Ingestion_timestamp", F.current_timestamp())
#               .withColumn("Ingestion_date", F.to_date("Ingestion_timestamp")))
#     df.write.format("delta").mode(mode).partitionBy("Ingestion_date").saveAsTable(silver_table)


# write_to_silver(df_accounts, accounts_silver)
# write_to_silver(df_organizations, organizations_silver)
# write_to_silver(df_signins, signins_silver)

# print("written to:", accounts_silver, organizations_silver, signins_silver, sep="\n  ")

In [0]:
df_accounts.write.format("delta").mode("overwrite").partitionBy("Ingestion_date").saveAsTable(accounts_silver)
df_organizations.write.format("delta").mode("overwrite").partitionBy("Ingestion_date").saveAsTable(organizations_silver)
df_signins.write.format("delta").mode("overwrite").partitionBy("Ingestion_date").saveAsTable(signins_silver)

In [0]:
# --- confirm the silver tables were written ---
for table_name in [accounts_silver, organizations_silver, signins_silver]:
    print(f"{table_name}: {spark.table(table_name).count()} rows")

coreview.silver.crm_accounts: 180 rows
coreview.silver.organizations: 12125 rows
coreview.silver.signins: 293842 rows
